# 05 · Recursion

**Module 05 – Functions** · Estimated time: 45 min

> 📘 The handbook covers recursion with a factorial example; the 📗 bootcamp doesn't have a dedicated lesson. This lesson expands it.

### 🎯 What you will learn
- What recursion is: a function that calls itself
- **Base case** and **recursive case**
- How the call stack works
- Classic examples: factorial, sum, power, Fibonacci, palindrome, reversing, flattening
- Recursion limits and `RecursionError`
- Memoization with `functools.cache` / `lru_cache`
- Recursion vs iteration: when to use which

## 1. The idea

A recursive function solves a problem by solving a **smaller version of the same problem**.

Every recursive function needs:
1. **Base case**: the simplest input, answered directly (stops the recursion)
2. **Recursive case**: calls itself with input that moves **towards** the base case

```
factorial(5) = 5 * factorial(4)
                    = 4 * factorial(3)
                           = 3 * factorial(2)
                                  = 2 * factorial(1)
                                         = 1          ← base case
```

In [1]:
def factorial(n):
    if n <= 1:                     # base case (also handles 0)
        return 1
    return n * factorial(n - 1)    # recursive case

print(factorial(5))
print([factorial(i) for i in range(8)])

120
[1, 1, 2, 6, 24, 120, 720, 5040]


## 2. Watching the call stack

In [2]:
def factorial_verbose(n, depth=0):
    indent = "  " * depth
    print(f"{indent}factorial({n}) called")
    if n <= 1:
        print(f"{indent}→ returns 1 (base case)")
        return 1
    result = n * factorial_verbose(n - 1, depth + 1)
    print(f"{indent}→ returns {result}")
    return result

factorial_verbose(4)

factorial(4) called
  factorial(3) called
    factorial(2) called
      factorial(1) called
      → returns 1 (base case)
    → returns 2
  → returns 6
→ returns 24


24

Each call waits on the **call stack** for the call below it to return. Then the results "unwind" back up.

## 3. More classic examples

In [3]:
def sum_list(nums):
    if not nums:                        # base: empty list
        return 0
    return nums[0] + sum_list(nums[1:])

def power(base, exp):
    if exp == 0:
        return 1
    return base * power(base, exp - 1)

def reverse(s):
    if len(s) <= 1:
        return s
    return reverse(s[1:]) + s[0]

def is_palindrome(s):
    if len(s) <= 1:
        return True
    if s[0] != s[-1]:
        return False
    return is_palindrome(s[1:-1])

def count_down(n):
    if n < 0:
        print("Liftoff!")
        return
    print(n, end=" ")
    count_down(n - 1)

print(sum_list([1, 2, 3, 4]), power(2, 10), reverse("python"), is_palindrome("racecar"))
count_down(5)

10 1024 nohtyp True
5 4 3 2 1 0 Liftoff!


### Where recursion really shines: nested / tree-shaped data

In [4]:
def flatten(items):
    result = []
    for item in items:
        if isinstance(item, list):
            result.extend(flatten(item))   # recurse into sub-lists
        else:
            result.append(item)
    return result

print(flatten([1, [2, [3, [4, 5]], 6], [[7]]]))

[1, 2, 3, 4, 5, 6, 7]


In [5]:
folder = {
    "name": "root", "size": 0,
    "children": [
        {"name": "a.txt", "size": 100, "children": []},
        {"name": "docs", "size": 0, "children": [
            {"name": "b.pdf", "size": 2000, "children": []},
            {"name": "c.md", "size": 50, "children": []},
        ]},
    ],
}

def total_size(node):
    return node["size"] + sum(total_size(child) for child in node["children"])

def show(node, depth=0):
    print("  " * depth + node["name"])
    for child in node["children"]:
        show(child, depth + 1)

print(total_size(folder))
show(folder)

2150
root
  a.txt
  docs
    b.pdf
    c.md


### Binary search (divide and conquer)

In [6]:
def binary_search(sorted_list, target, low=0, high=None):
    if high is None:
        high = len(sorted_list) - 1
    if low > high:
        return -1                      # not found
    mid = (low + high) // 2
    if sorted_list[mid] == target:
        return mid
    if sorted_list[mid] < target:
        return binary_search(sorted_list, target, mid + 1, high)
    return binary_search(sorted_list, target, low, mid - 1)

data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print(binary_search(data, 23), binary_search(data, 7))

5 -1


## 4. Fibonacci and the cost of naive recursion

In [7]:
calls = 0
def fib(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(25), "computed with", calls, "calls!")

75025 computed with 242785 calls!


The same sub-problems are computed again and again (exponential time). **Memoization** caches results:

In [8]:
from functools import cache       # 3.9+; lru_cache(maxsize=None) on older versions

@cache
def fib_fast(n):
    if n < 2:
        return n
    return fib_fast(n - 1) + fib_fast(n - 2)

print(fib_fast(100))
print(fib_fast.cache_info())

354224848179261915075
CacheInfo(hits=98, misses=101, maxsize=None, currsize=101)


## 5. Recursion limit

Python limits recursion depth (default ~1000) to protect against stack overflows.

In [9]:
import sys
print(sys.getrecursionlimit())

1000


In [10]:
def no_base_case(n):
    return no_base_case(n + 1)     # ❌ never stops

no_base_case(0)

RecursionError: maximum recursion depth exceeded

> 💡 You *can* raise the limit with `sys.setrecursionlimit()`, but deep recursion in Python is slow and risky. For deep problems, convert to a loop. Python does **not** do tail-call optimisation.

## 6. Recursion vs iteration

In [11]:
def factorial_iter(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

import math
print(factorial_iter(10), math.factorial(10))    # the built-in is fastest

3628800 3628800


| Use recursion when... | Use iteration when... |
|---|---|
| data is naturally nested (trees, folders, JSON) | it's a simple linear repetition |
| divide-and-conquer (merge sort, quicksort, binary search) | depth could be large (> ~1000) |
| backtracking (permutations, mazes, sudoku) | performance matters |

## 📝 Summary
- A recursive function calls itself; it **must** have a base case and make progress towards it.
- Great for nested/tree data and divide-and-conquer.
- Watch out for repeated work (use `@cache`) and the recursion limit.

## 🏋️ Practice
1. Write `count_digits(n)` recursively.
2. Write `gcd(a, b)` using Euclid's algorithm recursively (`gcd(a, b) = gcd(b, a % b)`).
3. Write `permutations(s)` that returns all permutations of a short string.
4. Write `depth(lst)` that returns the maximum nesting depth of a list (`[1, [2, [3]]]` → 3).
5. Write `hanoi(n, src, dst, via)` printing the moves for the Tower of Hanoi.

<details><summary>Show solutions</summary>

```python
def count_digits(n):
    n = abs(n)
    return 1 if n < 10 else 1 + count_digits(n // 10)

def gcd(a, b):
    return a if b == 0 else gcd(b, a % b)

def permutations(s):
    if len(s) <= 1:
        return [s]
    out = []
    for i, ch in enumerate(s):
        for p in permutations(s[:i] + s[i+1:]):
            out.append(ch + p)
    return out

def depth(lst):
    if not isinstance(lst, list):
        return 0
    return 1 + max((depth(x) for x in lst), default=0)

def hanoi(n, src="A", dst="C", via="B"):
    if n == 0:
        return
    hanoi(n - 1, src, via, dst)
    print(f"Move disk {n} {src} → {dst}")
    hanoi(n - 1, via, dst, src)
```
</details>